# 🚗 YOLOv8 Traffic Sign Detection for Autonomous Vehicles
### End-to-End Deep Learning Pipeline with YOLOv8 Nano

This Google Colab notebook implements training, evaluation, inference, and export for a 6-class traffic sign detection system for autonomous driving:
1. `left_turn` (Class 0)
2. `straight_ahead` (Class 1)
3. `right_turn` (Class 2)
4. `dead_end` (Class 3)
5. `no_entry` (Class 4)
6. `stop` (Class 5)


## Cell 1: Environment Setup & Package Installation

In [ ]:
# Install Ultralytics YOLO package
!pip install -q ultralytics

import os
import sys
import shutil
import zipfile
import glob
import yaml
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ultralytics
from ultralytics import YOLO

print("Ultralytics Version:", ultralytics.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Notice: Running on CPU. For fast training, enable GPU under Runtime -> Change runtime type.")


## Cell 2: Google Drive Mounting & Directory Setup

In [ ]:
# Mount Google Drive for persistent weights storage
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("Not in Colab or Drive mount skipped:", e)
    DRIVE_ROOT = "."

# Target output directory in Google Drive
PROJECT_OUTPUT_DIR = os.path.join(DRIVE_ROOT, "YOLOv8_Project")
os.makedirs(PROJECT_OUTPUT_DIR, exist_ok=True)

# Search for Autonomous-Car.zip in Drive or local /content
candidate_zip_paths = [
    os.path.join(DRIVE_ROOT, "Autonomous-Car.zip"),
    "/content/Autonomous-Car.zip",
    "Autonomous-Car.zip"
]

ZIP_PATH = None
for path in candidate_zip_paths:
    if os.path.exists(path):
        ZIP_PATH = path
        break

print("Project Output Directory:", PROJECT_OUTPUT_DIR)
print("Found Dataset Archive:", ZIP_PATH)


## Cell 3: Dataset Extraction to Fast Local Storage

In [ ]:
# Extract ZIP to local SSD (/content/dataset) for high-speed I/O during training
EXTRACT_DIR = "/content/dataset"
os.makedirs(EXTRACT_DIR, exist_ok=True)

if ZIP_PATH and os.path.exists(ZIP_PATH):
    print(f"Extracting {ZIP_PATH} to {EXTRACT_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Dataset extraction complete!")
else:
    print("Please upload Autonomous-Car.zip to /content/ or your Drive root before continuing.")


## Cell 4: Locate Dataset & Generate Colab data.yaml

In [ ]:
# Locate the primary 6-class dataset directory
search_paths = [
    os.path.join(EXTRACT_DIR, "Autonomous-Car", "Traffic-Signs-new"),
    os.path.join(EXTRACT_DIR, "Traffic-Signs-new"),
    "dataset"
]

DATASET_DIR = None
for p in search_paths:
    if os.path.isdir(p) and os.path.isdir(os.path.join(p, "train")):
        DATASET_DIR = os.path.abspath(p)
        break

if not DATASET_DIR:
    raise FileNotFoundError("Traffic-Signs-new dataset directory not found.")

print("Target Dataset Directory:", DATASET_DIR)

# Verify Dataset Counts (4200 / 1200 / 600)
for split in ["train", "valid", "test"]:
    img_count = len(os.listdir(os.path.join(DATASET_DIR, split, "images")))
    lbl_count = len(os.listdir(os.path.join(DATASET_DIR, split, "labels")))
    print(f"{split.upper():<6} -> Images: {img_count}, Labels: {lbl_count}")

# Write dynamic data.yaml with absolute Colab path to prevent path lookup failures
DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")
yaml_content = {
    "path": DATASET_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 6,
    "names": [
        "left_turn",
        "straight_ahead",
        "right_turn",
        "dead_end",
        "no_entry",
        "stop"
    ]
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_content, f, sort_keys=False)

print(f"\nUpdated {DATA_YAML} successfully!")


## Cell 5: Model Training on GPU with Google Drive Checkpoints
Trains YOLOv8 Nano for 100 epochs with automatic batch sizing and checkpoints saved directly to Google Drive.

In [ ]:
# Load base pre-trained model
model = YOLO("yolov8n.pt")

RUN_NAME = "traffic_signs_100_epochs"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Starting YOLOv8n Training...")
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
    patience=0,
    optimizer="auto",
    project=PROJECT_OUTPUT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    plots=True,
    verbose=True
)

BEST_WEIGHTS = os.path.join(PROJECT_OUTPUT_DIR, RUN_NAME, "weights", "best.pt")
LAST_WEIGHTS = os.path.join(PROJECT_OUTPUT_DIR, RUN_NAME, "weights", "last.pt")
print("Best model weights saved at:", BEST_WEIGHTS)
print("Last model weights saved at:", LAST_WEIGHTS)


## Cell 6: Benchmark Evaluation on Unseen Test Split

In [ ]:
# Load the trained model
eval_model = YOLO(BEST_WEIGHTS)

print("Evaluating on Test Split...")
metrics = eval_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    plots=True,
    device=DEVICE,
    verbose=True
)

print("\n--- TEST SPLIT EVALUATION METRICS ---")
print(f"Precision (P)   : {float(metrics.box.mp)*100:.2f}%")
print(f"Recall (R)      : {float(metrics.box.mr)*100:.2f}%")
print(f"mAP @ 0.50      : {float(metrics.box.map50)*100:.2f}%")
print(f"mAP @ 0.50-0.95 : {float(metrics.box.map)*100:.2f}%")


## Cell 7: Visualizing Confusion Matrix & Training Metrics

In [ ]:
run_dir = os.path.join(PROJECT_OUTPUT_DIR, RUN_NAME)
plots = ["results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for i, p_name in enumerate(plots):
    p_path = os.path.join(run_dir, p_name)
    if os.path.exists(p_path):
        axes[i].imshow(Image.open(p_path))
        axes[i].set_title(p_name, fontsize=14, fontweight="bold")
        axes[i].axis("off")
    else:
        axes[i].text(0.5, 0.5, f"{p_name} not found", ha="center")
        axes[i].axis("off")

plt.tight_layout()
plt.show()


## Cell 8: Sample Test Image Inference & Visual Badging

In [ ]:
test_images = sorted(glob.glob(os.path.join(DATASET_DIR, "test", "images", "*.jpg")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, img_path in enumerate(test_images):
    res = eval_model.predict(source=img_path, conf=0.40, verbose=False)[0]
    annotated_rgb = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(annotated_rgb)
    axes[i].set_title(f"Test Sample {i+1}", fontsize=12)
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## Cell 9: Model Export for Autonomous Car Embedded Platforms (ONNX)

In [ ]:
# Export trained weights to ONNX format
print("Exporting best model to ONNX for Jetson / Raspberry Pi / Edge deployment...")
onnx_file = eval_model.export(format="onnx", dynamic=False, simplify=True)
print("ONNX model saved to:", onnx_file)
